# Passaggio 7 sul test: controlli di robustezza

**Domanda.** Le differenze del [Passaggio 6 sul test](06_test_backtest.ipynb) tra B, C e la A sono vere o fortuna? Gli stessi controlli del train ([07_robustezza.ipynb](07_robustezza.ipynb)), decisi in [docs/Test.md](../docs/Test.md):

1. tabella finale: metriche, differenza dalla A con intervallo di confidenza (bootstrap sui giorni), beta e alpha rispetto a SPY;
2. costi 0 e 5;
3. dipendenza da una coppia o da un settore, e risultato senza PYPL-XYZ.

Ogni numero è accanto al suo valore del train. **Non si fanno** il confronto tra le due metà del periodo né il controllo senza il 13 agosto, che è un giorno del train.

Il notebook legge solo i file salvati in `Results/test/` da [test_finale.py](test_finale.py) e i prezzi di SPY del test (`python -m src.test_dati`).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, next(str(p) for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()))   # cartella del progetto: rende importabile src

import pandas as pd

from src.config import RESULTS, RESULTS_TEST, SPY_TEST, TEST_START, TEST_END
from src.backtest import load_trades, add_costs, daily_pnl, summary, save_results
from src.signals import list_pairs, load_pair
from src.varianti_agente.confronto import Variante, delta_giornaliero, bootstrap, beta_alpha, settori

pd.options.display.float_format = "{:,.2f}".format
pd.options.display.width = 220
A = load_trades(RESULTS_TEST / "trades_A.csv", allow_test=True)
B = load_trades(RESULTS_TEST / "trades_B.csv", allow_test=True)
C = load_trades(RESULTS_TEST / "trades_C.csv", allow_test=True)
d0 = load_pair(list_pairs().pair.iloc[0])
days = pd.DatetimeIndex(d0.ts.normalize().unique())
days = days[(days >= TEST_START) & (days <= TEST_END)]
rif = {"A": A}
varianti = [Variante("finbert_episodio", v, "A", True, "FinBERT, finestra dell'episodio", t) for v, t in (("B", B), ("C", C))]
spy = pd.read_parquet(SPY_TEST).set_index("date").close
mkt = spy.pct_change().reindex(days)                 # il rendimento del 17 settembre usa la chiusura del 16
assert mkt.notna().all(), "manca SPY in qualche giorno del test"
key = lambda v: (v.nome, v.versione)                  # noqa: E731
print(f"{len(days)} giorni di test; SPY: rendimento nel periodo {(spy.iloc[-1] / spy.iloc[0] - 1):.1%}, volatilità annua {mkt.std() * 252 ** 0.5:.1%}")

32 giorni di test; SPY: rendimento nel periodo 0.5%, volatilità annua 11.4%


## 1. Tabella finale (costo 0)

- metriche di A, B e C;
- `delta_usd`: differenza in dollari dalla A, con l'intervallo di confidenza al 95% ottenuto ricampionando i giorni del test con reinserimento (10.000 volte);
- `quota_sotto_zero`: la quota di ricampionamenti in cui la versione guadagna meno della A;
- beta e alpha annualizzato rispetto a SPY, con le statistiche t.

Il bootstrap tratta i giorni come indipendenti. Con 32 giorni gli intervalli sono larghi.

In [2]:
def metrics(t):
    s = summary(add_costs(t, 0), days)
    return {k: s[k] for k in ("trade", "net_bps", "win_rate", "profit_factor", "avg_minutes", "pnl_usd", "sharpe")} | dict(max_drawdown_pct=100 * s["max_drawdown"]) | \
           beta_alpha(daily_pnl(add_costs(t, 0), days).ret, mkt)

rows = [dict(variante="A", versione="") | metrics(A)]
rows += [dict(variante=v.nome, versione=v.versione) | metrics(v.trades) | bootstrap(delta_giornaliero(v, rif, days)) for v in varianti]
finale = pd.DataFrame(rows).set_index(["variante", "versione"])
train = pd.read_csv(RESULTS / "robustezza/tabella_finale.csv", index_col=[0, 1]).rename(index=lambda x: "" if pd.isna(x) else x, level=1)
train = train.loc[[("A", ""), ("finbert_episodio", "B"), ("finbert_episodio", "C")]]
cols = ["trade", "net_bps", "win_rate", "profit_factor", "sharpe", "max_drawdown_pct", "pnl_usd", "delta_usd", "ic95_basso", "ic95_alto", "quota_sotto_zero",
        "beta", "t_beta", "alpha_annuo", "t_alpha"]
pd.concat({"test": finale[cols], "train": train[cols]}, names=["periodo"])

trade  net_bps  win_rate  profit_factor  sharpe  max_drawdown_pct  pnl_usd  delta_usd  ic95_basso  ic95_alto  quota_sotto_zero  beta  t_beta  alpha_annuo  t_alpha
periodo variante         versione                                                                                                                                                                    
test    A                            500     4.94      0.63           1.31    4.92             -0.17 2,472.42        NaN         NaN        NaN               NaN -0.04   -2.19         0.07     1.91
        finbert_episodio B           493     4.63      0.63           1.29    4.73             -0.19 2,283.56    -188.86     -526.85     135.16              0.87 -0.04   -2.10         0.06     1.83
                         C           495     2.48      0.61           1.15    2.77             -0.21 1,229.96  -1,242.46   -2,182.51    -374.41              1.00 -0.04   -2.30         0.03     1.11
train   A                            973     7.44      0.60           1.45    8.13             -0.15 7,239.43        NaN         NaN        NaN               NaN  0.01    0.59         0.09     4.07
        finbert_episodio B           968     7.18      0.60           1.44    8.13             -0.14 6,954.50    -284.92   -1,598.48     834.27              0.64  0.01    0.62         0.09     4.07
                         C           969     6.80      0.60           1.42    7.44             -0.14 6,588.66    -650.77   -2,202.37     847.41              0.81  0.01    0.54         0.09     3.73

## 2. Costi

Differenza in dollari dalla A con 0 e 5 punti base per operazione. Ogni trade bloccato risparmia i costi: con costi alti anche un blocco a caso migliora il risultato, quindi un vantaggio che compare solo a costo 5 non vuol dire che l'agente scelga bene.

In [3]:
COSTS = (0, 5)
costi = pd.DataFrame({key(v): {c: summary(add_costs(v.trades, c), days)["pnl_usd"] - summary(add_costs(A, c), days)["pnl_usd"] for c in COSTS}
                      for v in varianti}).T.rename_axis(["variante", "versione"])
costi.columns = [f"delta_usd_costo_{c}" for c in COSTS]
costi_train = pd.read_csv(RESULTS / "robustezza/costi.csv", index_col=[0, 1]).loc["finbert_episodio"][costi.columns]
pd.concat({"test": costi, "train": pd.concat({"finbert_episodio": costi_train})}, names=["periodo"])

delta_usd_costo_0  delta_usd_costo_5
periodo variante         versione                                      
test    finbert_episodio B                   -188.86             -48.86
                         C                 -1,242.46          -1,142.46
train   finbert_episodio B                   -284.92            -184.92
                         C                   -650.77            -570.77

## 3. Dipende da una coppia o da un settore?

Differenza in dollari dalla A (costo 0) per coppia e per settore, accanto al train. Poi il risultato senza PYPL-XYZ, la coppia da cui sul train veniva quasi tutta la differenza.

In [4]:
sector = settori()

def delta_by_pair(v):
    p = lambda t: add_costs(t, 0).groupby("pair").pnl_usd.sum()   # noqa: E731
    return p(v.trades).sub(p(A), fill_value=0).reindex(sector.index, fill_value=0)

by_pair = pd.DataFrame({key(v): delta_by_pair(v) for v in varianti}).T.rename_axis(["variante", "versione"])
pairs_train = pd.read_csv(RESULTS / "robustezza/coppie.csv", index_col=[0, 1]).loc[["finbert_episodio"]]
pd.concat({"test": by_pair, "train": pairs_train[by_pair.columns]}, names=["periodo"]).T

periodo              test                    train          
variante finbert_episodio         finbert_episodio          
versione                B       C                B         C
MA-V                 0.00  -20.66             0.00   -137.26
CSX-UNP              0.00 -125.09             0.00      0.00
PYPL-XYZ             0.00 -390.95          -563.88 -1,482.55
HD-LOW               0.00    0.00             0.00      0.00
DHR-TMO            -25.25  -97.40             0.00    -99.47
CRM-WDAY             1.56 -259.93           297.67    653.49
JPM-WFC              0.00   25.63             0.00   -160.94
NXPI-TXN             0.00    0.00             0.00      0.00
ELV-UNH              0.00  118.39            31.05     82.44
CL-PG                0.00  -38.41             0.00      0.00
GS-MS                0.00   22.94             8.97    174.19
ISRG-SYK          -102.04 -154.44             0.00   -107.36
KO-PEP               0.00 -148.37             0.00    -66.57
BAC-C              -63.13 -216.21             0.00    134.82
COP-EOG              0.00   42.05           -58.74    358.44

In [5]:
by_sector = by_pair.T.groupby(sector).sum().T
sectors_train = pd.read_csv(RESULTS / "robustezza/settori.csv", index_col=[0, 1]).loc[["finbert_episodio"]]
pd.concat({"test": by_sector, "train": sectors_train.reindex(columns=by_sector.columns)}, names=["periodo"]).T

periodo                                    test                    train          
variante                       finbert_episodio         finbert_episodio          
versione                                      B       C                B         C
gruppo                                                                            
Alimenti, bevande e tabacco                0.00 -148.37             0.00    -66.57
Attrezzature e servizi sanitar          -102.04  -36.05            31.05    -24.92
Banche                                   -63.13 -190.59             0.00    -26.12
Beni discrezionali Distribuzio             0.00    0.00             0.00      0.00
Energia                                    0.00   42.05           -58.74    358.44
Farmaceutica, Biotecnologie e            -25.25  -97.40             0.00    -99.47
Prodotti per la casa e l'igien             0.00  -38.41             0.00      0.00
Semiconduttori e apparecchiatu             0.00    0.00             0.00      0.00
Servizi finanziari                         0.00 -388.67          -554.90 -1,445.62
Software e servizi                         1.56 -259.93           297.67    653.49
Trasporto                                  0.00 -125.09             0.00      0.00

In [6]:
senza = pd.DataFrame({"delta_usd": by_pair.sum(axis=1), "delta_PYPL_XYZ": by_pair["PYPL-XYZ"], "delta_senza_PYPL_XYZ": by_pair.drop(columns="PYPL-XYZ").sum(axis=1)})
senza_train = pd.DataFrame({"delta_usd": pairs_train.sum(axis=1), "delta_PYPL_XYZ": pairs_train["PYPL-XYZ"],
                            "delta_senza_PYPL_XYZ": pairs_train.drop(columns="PYPL-XYZ").sum(axis=1)})
pd.concat({"test": senza, "train": senza_train}, names=["periodo"])

delta_usd  delta_PYPL_XYZ  delta_senza_PYPL_XYZ
periodo variante         versione                                                 
test    finbert_episodio B           -188.86            0.00               -188.86
                         C         -1,242.46         -390.95               -851.50
train   finbert_episodio B           -284.92         -563.88                278.95
                         C           -650.77       -1,482.55                831.78

## 4. Salvataggio

In [7]:
for p in save_results({"test/robustezza/tabella_finale": finale, "test/robustezza/costi": costi, "test/robustezza/coppie": by_pair,
                       "test/robustezza/settori": by_sector, "test/robustezza/senza_PYPL_XYZ": senza}):
    print(p.relative_to(RESULTS.parent))

Results/test/robustezza/tabella_finale.csv
Results/test/robustezza/costi.csv
Results/test/robustezza/coppie.csv
Results/test/robustezza/settori.csv
Results/test/robustezza/senza_PYPL_XYZ.csv


## 5. Cosa ci dicono i controlli

**B: indistinguibile dalla A.** La B perde 189 dollari, con un intervallo al 95% **da -527 a +135**: contiene lo zero, quindi la differenza può essere fortuna. È sotto la A nell'87% dei ricampionamenti. Sul train l'intervallo era da -1.598 a +834.

**C: peggiore della A.** La C perde 1.242 dollari, con un intervallo **da -2.183 a -374**: è tutto sotto zero e la C è sotto la A in tutti i ricampionamenti. Sul test seguire la notizia peggiora la strategia con una certa sicurezza. Sul train l'intervallo conteneva ancora lo zero (da -2.202 a +847).

**Mercato.** Il beta di A, B e C rispetto a SPY è -0,04, con statistica t intorno a -2,2. In un periodo in cui SPY è salito dello 0,5%, vuol dire un legame molto piccolo con la borsa. Resta una strategia quasi neutrale al mercato, come sul train (beta 0,01). L'alpha annualizzato della A ha t = 1,9 (train 4,1): su 32 giorni riportati a un anno va letto con cautela.

**Costi.** A 5 punti base la B perde solo 49 dollari rispetto alla A: bloccando 7 trade risparmia i loro costi. È un risparmio di costi, non una scelta migliore. La C resta sotto di 1.142 dollari.

**Coppie e settori.**
- **B.** I 7 blocchi sono in tre coppie: ISRG-SYK (-102), BAC-C (-63) e DHR-TMO (-25), più +2 su CRM-WDAY. Sul test l'agente non blocca niente su PYPL-XYZ. Togliere quella coppia non cambia la B (-189), mentre sul train la rendeva positiva (+279). La spiegazione del train ("la differenza viene da un solo trade su PYPL-XYZ") sul test non vale: la B perde su più coppie.
- **C.** Perde in 9 coppie su 15 e guadagna in 4. Senza PYPL-XYZ perde ancora 852 dollari (train +832). La perdita è distribuita, non legata a una coppia.

**In sintesi.** I controlli confermano il Passaggio 6 sul test. Sul test la B non si distingue dalla A e la C è peggiore della A. Né bloccare né seguire le notizie con `finbert_episodio` migliora la strategia di pairs trading.

**Limiti.**
- Il bootstrap tratta i giorni come indipendenti, e i giorni sono solo 32.
- Gli intervalli sono più stretti che sul train perché il periodo è più breve e la A guadagna meno, non perché l'agente sia più prevedibile.